In [ ]:
# !pip install anthropic

In [ ]:
from pipeline_helpers import BQ_DS, gcs_bq_map, req_fields_map

In [ ]:
#PROJECT NAV BASICS
PROJECT_ID = !(gcloud config get-value core/project)
PROJECT_ID = PROJECT_ID[0]
REGION = "us-central1"
# DOC_BUCKET_DIR = "loan-pipeline-demo-financial-statements-bucket"
DOC_BUCKET_DIR = "loan-pipeline-demo-appraisals-bucket"
PIPELINE_END_TABLE = gcs_bq_map[DOC_BUCKET_DIR]

In [ ]:
#PACKAGE IMPORTS
import os
import json
import pandas as pd
from google.cloud import storage, bigquery
import anthropic

In [ ]:
#DEFINE CLIENTS
client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
storage_client = storage.Client(project=PROJECT_ID)
bq_client = bigquery.Client(project=PROJECT_ID)

In [ ]:
# Fields specific to this doc type (financial statements).
# NOTE: loan_number and statement_period are handled separately inside
# parse_document() since every doc type will need those regardless of
# which content fields vary — keeps this function reusable for
# appraisals/invoices later, you'll just swap this list.
REQUIRED_FIELDS = req_fields_map[DOC_BUCKET_DIR]

In [ ]:
#LLM CALL + JSON PARSING FUNCTION
def call_llm_for_extraction(doc_text: str, required_fields: list[tuple[str, str]]) -> dict:
    """
    Sends document text to the LLM and asks for a strict JSON response
    containing every field in required_fields (a list of (field_name, dtype)
    tuples) with a confidence score. No fields are special-cased — identifiers
    like loan_number and document_date are just STRING-typed entries in the
    same list as FLOAT-typed content fields.
    """
    field_list_str = "\n".join(f"- {name}: {dtype}" for name, dtype in required_fields)

    system_prompt = (
        "You are a document parser for a real estate lending underwriting system. "
        "You will be given the raw text of an unstructured lending document — "
        "this may be a financial statement, appraisal, invoice, or similar. "
        "Extract the requested fields and return ONLY a valid JSON object — "
        "no preamble, no markdown code fences, no explanation text.\n\n"
        f"Fields to extract (name: type):\n{field_list_str}\n\n"
        "For each field, return an object with:\n"
        "  - value: the extracted value. For STRING fields, return the exact "
        "text as written in the document. For FLOAT fields, return the "
        "numeric value ONLY (strip $, commas, % signs — just the raw number "
        "as a float).\n"
        "  - confidence: your confidence in this extraction, 0-100\n\n"
        "If a field cannot be found, set value to null and confidence to 0.\n\n"
        "Expected JSON shape:\n"
        "{\n"
        '  "loan_number": {"value": "MTN-10234", "confidence": 99},\n'
        '  "document_date": {"value": "T-12 Ending 12/31/2025", "confidence": 95},\n'
        '  "total_revenue": {"value": 2845600, "confidence": 95},\n'
        "  ...\n"
        "}"
    )

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        system=system_prompt,
        messages=[{"role": "user", "content": doc_text}],
    )

    raw_text = response.content[0].text.strip()
    # Defensive: strip markdown code fences if the model adds them despite
    # instructions not to — cheap insurance against a brittle parse failure.
    if raw_text.startswith("```"):
        raw_text = raw_text.strip("`")
        raw_text = raw_text.replace("json", "", 1).strip()
    return json.loads(raw_text)

In [ ]:
#PER-DOC PARSE FUNCTION
def parse_document(blob_name: str, doc_text: str, required_fields: list[tuple[str, str]]) -> dict:
    """
    Takes a document's text content and returns a flattened dict ready to
    append as one row in the output table. Every field in required_fields
    (including identifiers like loan_number and document_date) is handled
    uniformly — no special-casing.
    """
    extracted = call_llm_for_extraction(doc_text, required_fields)

    row = {"source_filename": blob_name}

    for field_name, _dtype in required_fields:
        field_data = extracted.get(field_name, {"value": None, "confidence": 0})
        row[field_name] = field_data.get("value")
        row[f"{field_name}_confidence"] = field_data.get("confidence")

    return row

In [ ]:
#LOOP OVER GCS BUCKET, BUILD RECORDS, ASSEMBLE DF
bucket = storage_client.bucket(DOC_BUCKET_DIR)
blobs = list(bucket.list_blobs())

records = []
failures = []

for blob in blobs:
    if not blob.name.endswith(".txt"):
        continue  # skip any non-doc files sitting in the bucket

    try:
        doc_text = blob.download_as_text()  # read straight into memory, no temp file
        row = parse_document(blob.name, doc_text, REQUIRED_FIELDS)
        records.append(row)
        print(f"Parsed: {blob.name}")
    except Exception as e:
        # Don't let one bad doc kill the whole run — log it and keep going.
        failures.append({"filename": blob.name, "error": str(e)})
        print(f"FAILED: {blob.name} — {e}")

df = pd.DataFrame(records)

if failures:
    print(f"\n{len(failures)} document(s) failed to parse:")
    for f in failures:
        print(f"  - {f['filename']}: {f['error']}")

df

In [ ]:
#PUSH DF TO BQ TABLE
#NOTE WE'RE NOT APPENDING BUT REPLACING DURING DEV
table_id = f"{PROJECT_ID}.{BQ_DS}.{PIPELINE_END_TABLE}"

schema = [
    bigquery.SchemaField("source_filename", "STRING"),
]
for field_name, dtype in REQUIRED_FIELDS:
    schema.append(bigquery.SchemaField(field_name, dtype))
    schema.append(bigquery.SchemaField(f"{field_name}_confidence", "FLOAT"))

job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition="WRITE_TRUNCATE",  # overwrite on re-run during dev;
                                          # switch to WRITE_APPEND once stable
)

load_job = bq_client.load_table_from_dataframe(df, table_id, job_config=job_config)
load_job.result()  # wait for completion

print(f"Loaded {len(df)} rows into {table_id}")